# Notebook 1: Data sampling, cleaning, labels, and EDA
Group Theta (seed 49). Draws our 50,000-review sample from the Amazon Fine Food Reviews file, cleans it, derives three-class labels, and produces Figure 1 for Milestone 1. Run from the repository root or from `notebooks/`.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

SEED = 49  # Group Theta's seed from the course seed sheet
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "Reviews.csv"          # download instructions in data/README.md
FIG_DIR = ROOT / "report" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

full = pd.read_csv(DATA, usecols=["Id", "Score", "Text", "Summary"])
print("Full dataset rows:", len(full))
sample = full.sample(n=50000, random_state=SEED).reset_index(drop=True)
print("Sample rows:", len(sample))
print("Missing values in the sample:")
print(sample[["Score", "Text", "Summary"]].isna().sum())

In [ ]:
clean = sample.dropna(subset=["Text"]).copy()
clean["Summary"] = clean["Summary"].fillna("")
n_dup = clean.duplicated(subset=["Text"]).sum()
clean = clean.drop_duplicates(subset=["Text"], keep="first").reset_index(drop=True)
print("Duplicate review texts removed:", n_dup)
print("Rows after cleaning:", len(clean))

def to_label(score):
    if score <= 2:
        return "negative"
    if score == 3:
        return "neutral"
    return "positive"

clean["label"] = clean["Score"].apply(to_label)
clean["n_words"] = clean["Text"].astype(str).str.split().str.len()
print("\nClass distribution (%):")
print((clean["label"].value_counts(normalize=True) * 100).round(2))
print("\nAverage review length (words):", round(clean["n_words"].mean(), 1))
print("Median review length (words):", clean["n_words"].median())

In [ ]:
counts = sample["Score"].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 4))
counts.plot(kind="bar", color="steelblue", ax=ax)
ax.set_xlabel("Star rating")
ax.set_ylabel("Number of reviews")
ax.set_title("Star rating distribution (Group Theta, seed 49)")
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=8)
plt.xticks(rotation=0)
plt.tight_layout()
fig.savefig(FIG_DIR / "fig1_score_distribution.png", dpi=200)
plt.show()